# 16 — mélange par étude

Le réseau renvoie le mélange. Pour chaque cible, un softmax donne un poids par modèle. Les poids somment à 1. La probabilité de la cible est la somme des poids fois la probabilité du modèle.

Série unique, la même pour chaque étude : le **sagittal fat-sat** (`SAG_FS`). C'est le plan des ménisques, du LCA, de l'épanchement, du kyste de Baker et de la contusion. Il est lisible sur 2946 des 3142 études. L'axial fat-sat est présent presque partout, et il voit surtout le compartiment fémoro-patellaire. Quand le sagittal manque, le vecteur est nul : le gate retombe sur le biais. Huit coupes fixes, canon gauche, ConvNeXt-Tiny ImageNet gelé, puis une PCA ajustée sur le fit.

Trois gates. Même loss : BCE où chaque cible pèse autant, et où les positifs pèsent autant que les négatifs. Même checkpoint : l'epoch de meilleure macro-AUC **train**. La val (652) et le gold (58) ne choisissent pas l'epoch.

- `constant` — 84 biais, un mélange par cible, identique pour toutes les études.
- `logits+slots` — la matrice du notebook 13, ramenée dans le simplexe (logits complets et six slots, par cible).
- `SAG_FS` — logit diagonal, slots partagés entre les cibles, 4 axes de la série. Pénalité L2 plus forte sur ces résidus.

Sur le pool ci-dessous, avec le distil dans la liste (fit = 1327 études qu'il n'a pas vues), le checkpoint qui tient est le constant : val 0.9661, gold 0.9197, epoch 37. La moyenne uniforme est à 0.9628 / 0.9180. Le meilleur modèle seul, v20h-distil, est à 0.9609 / 0.9160. Le gain de val est surtout l'ACL (+0.016).

`logits+slots` monte le train à 0.968 et le checkpoint finit à 0.9622 / 0.9137. `SAG_FS` monte le train à 0.967 et finit à 0.9616 / 0.9177. Le mélange à garder est `data/hybrid_pred/mix_gate_constant.npz`.

Commenter une ligne de `MODELS` retire le modèle. Si le distil reste dans la liste, les études de son `train_uids.txt` sortent du fit.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

ROOT = Path.cwd() if (Path.cwd() / "checkpoints").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.ensemble.mix_gate import (
    apply_mix,
    fit_mix_gate,
    load_frame,
    save_bundle,
    split_auc,
)
from src.ensemble.sag_embed import extract_sag_fs, load_embed
from src.nlp.lexicon import TARGETS
from src.train.metrics import macro_auc

PRED = ROOT / "data" / "hybrid_pred"
CACHE = PRED / "sag_fs_embed.npz"
REBUILD_EMBED = False

MODELS = [
    ("ViT-k320", "hybrid_preds_unseen_rsna_vitkllrd08.csv"),
    ("v20h", "hybrid_preds_unseen_rsna__subset_v20h_dino_ep11_v3b_hybrid025.csv"),
    ("v20h-c145", "hybrid_preds_unseen_rsna__subset_v20h_dino_ep11_v3b_hybrid025_wd01_crop145.csv"),
    ("CoAt-a", "hybrid_preds_unseen_rsna__subset_coat_a_llrd08_v3b_hybrid025.csv"),
    ("CNX-r7", "hybrid_preds_unseen_rsna__subset_cnx_r7_in22k_v3b_hybrid025.csv"),
    ("v20h-distil", "hybrid_preds_unseen_rsna__subset_v20h_dino_ep11_v3b_rsna1687_distil_main_crop145.csv"),
    ("v20b", "hybrid_preds_unseen_rsna__subset_v20b_dino_ep11_v3b_hybrid025.csv"),
]
DISTIL_UIDS = (
    ROOT / "checkpoints"
    / "subset_v20h_dino_ep11_v3b_rsna1687_distil_main_crop145"
    / "train_uids.txt"
)

EPOCHS = 400
LR = 0.05
WEIGHT_DECAY = 1e-2
SEED = 42

frame = load_frame(ROOT, MODELS, DISTIL_UIDS)
print("ROOT", ROOT)
print("modèles", frame["names"])
print(
    f"études {len(frame['uids'])} | fit {int(frame['fit_mask'].sum())} | "
    f"val {int(frame['val_mask'].sum())} | gold {int(frame['gold_mask'].sum())}"
)
if frame["use_distil"]:
    print(f"distil dans l'ensemble : {frame['fit_blocked']} études retirées du fit")


In [ ]:
if REBUILD_EMBED or not CACHE.exists():
    extract_sag_fs(frame["uids"], ROOT, out_path=CACHE)
embed, present = load_embed(CACHE, frame["uids"])
meta = np.load(CACHE)
print(str(meta["encoder"]), "|", str(meta["zip_path"]), "| coupes", int(meta["n_slices"]))
print(f"vecteur {embed.shape[1]} | SAG_FS lue {int(present.sum())}/{len(present)}")
print(
    "absente au fit",
    int((frame["fit_mask"] & ~present).sum()),
    "| val",
    int((frame["val_mask"] & ~present).sum()),
    "| gold",
    int((frame["gold_mask"] & ~present).sum()),
)


In [ ]:
rows = []
for i, name in enumerate(frame["names"]):
    rows.append({
        "mélange": name,
        "epoch": "",
        "paramètres": "",
        **split_auc(frame["y"], frame["probs"][:, i], frame),
    })
rows.append({
    "mélange": "uniforme",
    "epoch": "",
    "paramètres": "",
    **split_auc(frame["y"], frame["probs"].mean(axis=1), frame),
})

VARIANTS = [
    ("constant", dict(use_pred=False, use_empty=False, use_image=False)),
    ("logits+slots", dict(
        use_pred=True, pred_mode="full",
        use_empty=True, empty_mode="full",
        use_image=False,
    )),
    ("SAG_FS", dict(
        use_pred=True, pred_mode="diagonal",
        use_empty=True, empty_mode="shared",
        use_image=True, residual_decay=0.1, pca_dim=4,
    )),
]
bundles = {}
for name, flags in VARIANTS:
    bundle = fit_mix_gate(
        frame["probs"], frame["y"],
        frame["fit_mask"], frame["val_mask"], frame["gold_mask"],
        frame["empty"], embed, present,
        model_names=frame["names"],
        epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, seed=SEED,
        **flags,
    )
    bundles[name] = bundle
    save_bundle(PRED / f"mix_gate_{name.replace('+', '_')}.npz", bundle)
    m = bundle["metrics"]
    rows.append({
        "mélange": name,
        "epoch": bundle["config"]["best_epoch"],
        "paramètres": bundle["config"]["n_param"],
        "fit": m["train"],
        "val": m["val"],
        "gold": m["gold"],
    })
    print(
        f"{name:16}  epoch {bundle['config']['best_epoch']:3d}  "
        f"param {bundle['config']['n_param']:4d}  "
        f"AUC {m['train']:.4f} / {m['val']:.4f} / {m['gold']:.4f}",
        flush=True,
    )

table = pd.DataFrame(rows).set_index("mélange")
table.round(4)


In [ ]:
kept = bundles["constant"]
val = frame["val_mask"]
_, per_mix = macro_auc(frame["y"][val], kept["mixed"][val], list(TARGETS))
_, per_uni = macro_auc(frame["y"][val], frame["probs"].mean(1)[val], list(TARGETS))
delta = pd.Series(
    {t: per_mix[t] - per_uni[t] for t in TARGETS},
    name="val, constant moins uniforme",
)
print(delta.round(4).to_string())

w = pd.DataFrame(kept["mean_weight_fit"], index=list(TARGETS), columns=frame["names"])
print("\npoids du mélange constant, fit — identiques pour chaque étude")

fig, ax = plt.subplots(figsize=(8.2, 5.2))
im = ax.imshow(
    w.to_numpy(), vmin=0, vmax=max(0.35, float(w.to_numpy().max())),
    cmap="viridis", aspect="auto",
)
ax.set_xticks(range(len(frame["names"])))
ax.set_xticklabels(frame["names"], rotation=30, ha="right")
ax.set_yticks(range(len(TARGETS)))
ax.set_yticklabels(list(TARGETS))
ax.set_title("Mélange constant")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

i = int(np.flatnonzero(val)[0])
one_w, one_p = apply_mix(
    kept, frame["probs"][i:i + 1], frame["empty"][i:i + 1],
)
print("somme des poids", float(one_w.sum(-1).mean()))
w.round(3)
